# Award Topics: BERT Inference

Runs topic classification on awards from `award_topics_input` using the same fine-tuned BERT model as the works pipeline (`OpenAlex/bert-base-multilingual-cased-finetuned-openalex-topic-classification-title-abstract`). Predicts top 3 topics per award from `display_name` + `description`.

Output is appended to `award_topics_lm_output`, then processed rows are deleted from the input table. Runs on the same GPU cluster type as the works topics job (g5.12xlarge with A10G GPUs).

Mirrors `topics_inference.ipynb` for the awards pipeline (oxjob #123.1). Key differences vs the works version:
- `work_id` → `award_id`
- `title` → `display_name`
- `abstract` → `description`
- No `journal_name` column (awards have no journal).

Note: the model was fine-tuned on academic title+abstract pairs. Grant abstracts have a somewhat different register (future-tense, broader scope). Topic-quality is expected to be acceptable based on the same text-only inputs the active works pipeline uses; sanity-sample after the backfill before unblocking the frontfill follow-up job (see ACCEPTANCE.md Test 6).

In [ ]:
from transformers import TFAutoModelForSequenceClassification, pipeline, AutoTokenizer
import torch
import os

print(f"CUDA device count: {torch.cuda.device_count()}")

BATCH_SIZE = 25  # Matches works pipeline; tuned for g5.12xlarge A10G (24GB VRAM)
MODEL_PATH = "/Volumes/openalex/works/models/topic-classification-title-abstract"

class ModelCache:
    model = None

    @classmethod
    def load(cls):
        if cls.model is not None:
            return

        num_devices = torch.cuda.device_count()
        if num_devices == 0:
            cls.assigned_device = -1
        else:
            cls.assigned_device = os.getpid() % num_devices

        cls.model = pipeline(
            task="text-classification",
            model=MODEL_PATH,
            device=cls.assigned_device,
            top_k=3,
            batch_size=BATCH_SIZE,
            truncation=True,
            max_length=512,
        )

### Load input data and cache

In [ ]:
df = (spark.table("openalex.awards.award_topics_input")
      .select("award_id", "display_name", "description")
      .limit(3840000)
)

# Dynamic repartitioning: ~2500 rows per partition, min 16 (GPU utilization), max 1024
row_count = df.count()
num_partitions = max(16, min(1024, row_count // 2500))
df = df.repartition(num_partitions)

print(f"Input Row Count: {row_count}")
print(f"Using {num_partitions} partitions (~{row_count // num_partitions} rows per partition)")

### Run inference

In [ ]:
import json
from pyspark.sql.functions import *
from pyspark.sql.types import *
from topic_predictor import *

import time
import re

def process_partition(rows):
    """
    Process a partition using mapPartitions.
    Returns tuples to avoid driver memory pressure.
    """
    ModelCache.load()
    model = ModelCache.model

    batch_rows = []
    batch_texts = []

    def yield_batch(rows_batch, texts_batch):
        try:
            batch_outputs = model(texts_batch)
        except Exception as e:
            print(f"Error processing batch: {e}")
            batch_outputs = [[] for _ in texts_batch]

        for row, output in zip(rows_batch, batch_outputs):
            lm_output = [
                {
                    "topic_id": 10000 + int(topic["label"].split(":")[0]),
                    "score": float(topic["score"]),
                }
                for topic in output
            ] if output else None

            yield (row.award_id, row.display_name, row.description, lm_output)

    for row in rows:
        if row is None:
            continue

        # Reuse the works text cleaners verbatim (strip HTML, reject heavily non-Latin)
        display_name = clean_title(row.display_name) or ""
        description = clean_abstract(row.description) or ""

        # Skip scoring if >50% of text was stripped (non-Latin dominant) —
        # better to assign no topic than a garbage prediction from leftover fragments
        display_name_stripped = is_heavily_stripped(row.display_name, display_name)
        description_stripped = is_heavily_stripped(row.description, description)
        if display_name_stripped and (description_stripped or not row.description):
            yield (row.award_id, row.display_name, row.description, None)
            continue

        full_text = f"[CLS]<TITLE> {display_name.strip()} <ABSTRACT> {description.strip()} [SEP]"

        batch_rows.append(row)
        batch_texts.append(full_text)

        if len(batch_texts) >= BATCH_SIZE:
            yield from yield_batch(batch_rows, batch_texts)
            batch_rows = []
            batch_texts = []

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if batch_texts:
        yield from yield_batch(batch_rows, batch_texts)

start_time = time.time()

topic_struct = StructType([
    StructField("topic_id", IntegerType(), True),
    StructField("score", FloatType(), True),
])

output_schema = StructType([
    StructField("award_id", LongType(), True),
    StructField("display_name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("lm_topics", ArrayType(topic_struct), True),
])

res_rdd = df.select("award_id", "display_name", "description").rdd.mapPartitions(process_partition)
res_df = spark.createDataFrame(res_rdd, output_schema).cache()
output_count = res_df.count()
print(f"Output Row count: {output_count}")

res_df = (res_df.select("award_id", "display_name", "description", "lm_topics")
                .withColumn("lm_primary_topic", col("lm_topics")[0])
                .withColumn("source", lit("bert_lm"))
                .withColumn("created_timestamp", current_timestamp()))
res_df.write.mode("append").saveAsTable("openalex.awards.award_topics_lm_output")

runtime = time.time() - start_time
print(f"Total runtime: {runtime:.4f} seconds")
print(f"Total throughput: {output_count / runtime:.4f} inferences/sec")

res_df.createOrReplaceTempView("res_df_temp")

spark.sql("""
    DELETE FROM openalex.awards.award_topics_input
    WHERE award_id IN (SELECT award_id FROM res_df_temp)
""")
print("Removed processed award_ids from award_topics_input")